In [ ]:
# =========================
# sample complete code combined
# =========================
!pip install pdfplumber
import re
import json
import pandas as pd
import pdfplumber
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity



icd_df = pd.read_excel("ICD_code_Assignment.xlsx")
cpt_df = pd.read_excel("cpt_code_assignment.xlsx")

icd_df = icd_df[['ICD Code', 'Description']]
icd_df.columns = ['Code', 'Description']

cpt_df = cpt_df[['CPT Code', 'Description']]
cpt_df.columns = ['Code', 'Description']



model = SentenceTransformer("all-MiniLM-L6-v2")
icd_embeddings = model.encode(icd_df["Description"].tolist())
cpt_embeddings = model.encode(cpt_df["Description"].tolist())


def retrieve_codes(query, embeddings, df, top_k=5):
    q_emb = model.encode([query])
    scores = cosine_similarity(q_emb, embeddings)[0]
    idx = scores.argsort()[-top_k:][::-1]
    return df.iloc[idx]



def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            if page.extract_text():
                text += page.extract_text() + "\n"
    return text


def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()



ANATOMY_LIST = [
    "anal canal", "rectum", "sigmoid colon", "descending colon",
    "splenic flexure", "transverse colon", "hepatic flexure",
    "ascending colon", "cecum", "terminal ileum"
]


def extract_anatomy(text):
    return [a.title() for a in ANATOMY_LIST if a in text]


def extract_explicit_icds(text):
    return list(set(re.findall(r'\b[A-Z]\d{2}\.\d{1,2}\b', text)))


def infer_clinical_terms(text):
    terms = []

    if "colon cancer screening" in text or "z12.11" in text:
        terms.append("Colon cancer screening")

    if "hemorrhoids" in text:
        terms.append("Hemorrhoids")

    if "sessile polyp" in text:
        terms.append("Sessile polyp")

    if "cold snare" in text:
        terms.append("Polyp removal")

    if "no immediate complication" in text:
        terms.append("No immediate complication")

    return terms


def infer_procedures(text):
    procs = []
    if "colonoscopy" in text:
        procs.append("Colonoscopy")
    if "cold snare" in text:
        procs.append("Cold snare polypectomy")
    if "biopsy" in text and "cold snare" not in text:
        procs.append("Biopsy")
    return procs


def infer_cpt(text):
    if "cold snare" in text:
        return ["45385"]
    if "biopsy" in text:
        return ["45380"]
    if "colonoscopy" in text:
        return ["45378"]
    return []


def infer_hcpcs(text):
    if "propofol" in text or "lidocaine" in text:
        return ["J3490", "A4216"]
    return []



def process_clinical_input(pdf_path=None, raw_text=None):
    if not pdf_path and not raw_text:
        raise ValueError("Provide pdf_path or raw_text")

    text = extract_text_from_pdf(pdf_path) if pdf_path else raw_text
    text = preprocess(text)

    clinical_terms = infer_clinical_terms(text)
    anatomy = extract_anatomy(text)
    diagnosis = clinical_terms.copy()
    procedures = infer_procedures(text)

    explicit_icds = extract_explicit_icds(text)

    if explicit_icds:
        icd_codes = explicit_icds
    else:
        icd_candidates = retrieve_codes(
            " ".join(clinical_terms), icd_embeddings, icd_df, top_k=5
        )
        icd_codes = icd_candidates["Code"].tolist()

    cpt_codes = infer_cpt(text)
    hcpcs_codes = infer_hcpcs(text)

    return {
        "Clinical Terms": clinical_terms,
        "Anatomical Locations": anatomy,
        "Diagnosis": diagnosis,
        "Procedures": procedures,
        "ICD-10": icd_codes,
        "CPT": cpt_codes,
        "HCPCS": hcpcs_codes
    }




# --- PDF input ---
# result = process_clinical_input(pdf_path="Input data for Assignment.pdf")

# --- OR text input ---
sample_text = """
Diagnosis:
Pre-operative Diagnosis
R07.89 - Atypical chest pain R10.11 - Right upper quadrant abdominal pain
Post-operative Diagnosis
R07.89 - Atypical chest pain R10.11 - RUQ pain K29.70 - Gastritis
Procedures:
Procedure Code EGD w/Biopsy
Anesthesia Type: Monitored Anesthesia Care
Lactated Ringers - Solution, Intravenous as directed - 200 00, Last Administered By:
Smith, George At 1419 on 07/07/2025 Lidocaine HCI 2% Solution, IV - 60 00 , Last Administered By: Smith, George At 1408 on 07/07/2025 Propofol 500 MG/50ML Emulsion, Intravenous - 350 00, Last Administered By: Smith, George At 1419 on 07/07/2025
EGD PROCEDURE: There was nothing precluding endoscopy on history or physical exam. Informed consent was obtained with risks and benefits explained to the patient.
The patient tolerated the procedure well. There were no immediate complications The patient was placed in the left lateral decubitus position. The Olympus endoscope was inserted into the esophagus under direct visualization. It was advanced through the esophagus, into the stomach and through the pylorus to the duodenal bulb and 2nd portion of the duodenum. Careful inspection was made as the endoscope was removed including retroflexion in the stomach. Findings- In the distal esophagus there was an irregular Z-line from 38-39 cm suggestive of Barrett's esophagus. This was examined both white light and narrow band imaging. Biopsies were obtained of the distal esophagus. In the stomach there was mild antral and body gastritis. Biopsies were obtained for H.pylori. There were no ulcers or masses seen. The visualized portion of the duodenal appeared normal without ulcers or inflammation. IMPRESSION: The patient is 55-year-old male with atypical chest pain and right upper quadrant pain. Biopsies today were obtained for H.pylori and Barrett's esophagus. If the patient does Barrett's esophagus and a repeat EGD with biopsy in 6 months will be recommended.
Post Operative Impression
EGD The patient tolerated the procedure without complications. The EGD was uneventful.
"""

result = process_clinical_input(raw_text=sample_text)

print(json.dumps(result, indent=2))
